# 01 – Explore OSM restaurants

**Goal:** find out how McDonald's, Max, Burger King and Sibylla are tagged in OpenStreetMap, so we can match them reliably in dbt.

**Data:** `raw.osm_food_places`, loaded by `hamburgarkartan.geofabrik` from the Geofabrik Sweden extract of **2026-10-26**.

Run `uv run python -m hamburgarkartan.geofabrik` first.

In [ ]:
import duckdb
from hamburgarkartan.geofabrik import DB_PATH

# read_only = we only look, and other programs can still use the database
con = duckdb.connect(str(DB_PATH), read_only=True)

## 1. How much data?

In [ ]:
# all fast food places and restaurants in Sweden (+ chain brands with other tags)
con.sql("SELECT count(*) AS food_places FROM raw.osm_food_places")

**Finding:** 15 483 food places. Extraction took **28 s** (two-pass read).

## 2. Points or buildings?
A place can be a point (`node`) or drawn as a building (`way`). Buildings have no coordinates of their own – the script uses the centre of their corners.

In [ ]:
con.sql("""
    SELECT
        osm_type,
        count(*) AS places,
        count(*) FILTER (WHERE lat IS NULL) AS missing_coords  -- should be 0
    FROM raw.osm_food_places
    GROUP BY osm_type
""")

**Finding:** fill in – how many are buildings, and are any coordinates missing?

## 3. What did the brand filter add?
Besides `amenity = fast_food / restaurant`, the script keeps any place with a chain brand. These rows were caught **only** by the brand filter.

In [ ]:
con.sql("""
    SELECT osm_type, osm_id,
           tags['brand'] AS brand, tags['name'] AS name,
           tags['amenity'] AS amenity, tags['shop'] AS shop
    FROM raw.osm_food_places
    WHERE coalesce(tags['amenity'], '') NOT IN ('fast_food', 'restaurant')
""")

**Finding:** 5 extra places.
- 4 are real chain restaurants with the `amenity` tag missing → the brand filter was worth it.
- 1 is a **Max clothing shop** (`shop = clothes`) → false positive, must be removed later.

## 4. How are the four chains tagged?
`brand:wikidata` is a fixed ID per brand – more reliable than the name, which is spelled in many ways.

In [ ]:
con.sql("""
    WITH guessed AS (
        SELECT
            CASE
            WHEN lower(coalesce(tags['brand'], tags['name'])) LIKE '%mcdonald%'
              OR lower(coalesce(tags['brand'], tags['name'])) LIKE '%mc donald%' THEN 'mcdonalds'
            WHEN lower(coalesce(tags['brand'], tags['name'])) LIKE '%burger king%' THEN 'burger_king'
            WHEN lower(coalesce(tags['brand'], tags['name'])) LIKE '%sibylla%' THEN 'sibylla'
            WHEN lower(tags['brand']) IN ('max', 'max burgers')
              OR lower(tags['name']) IN ('max', 'max burgers')
              OR lower(tags['name']) LIKE 'max %' THEN 'max'
        END AS chain_guess,
            trim(tags['brand:wikidata']) AS brand_wikidata,
            tags['amenity'] AS amenity,
            tags['shop'] IS NOT NULL AS has_shop
        FROM raw.osm_food_places
    )
    SELECT chain_guess, brand_wikidata, amenity, has_shop, count(*) AS places
    FROM guessed
    WHERE chain_guess IS NOT NULL
    GROUP BY 1, 2, 3, 4
    ORDER BY chain_guess, places DESC
""").show(max_rows=60)

**Finding:** each chain has one Wikidata ID.

| Chain | `brand:wikidata` | With ID | Without ID |
|---|---|---|---|
| McDonald's | Q38076 | 199 | 5 |
| Max | Q1912172 | 164 | 7 |
| Burger King | Q177054 | 113 | 2 |
| Sibylla | Q488643 | 107 | 14 |

- **95 %** of chain places have the ID → use it as the main rule.
- 28 places (5 %) need a fallback on brand/name. Sibylla is tagged worst.
- The Max clothing shop has a **different** ID (Q24939079) → the ID tells the two Max apart.

## 5. Can we trust the name fallback?
Check the Max places without an ID – the pattern `max %` could catch other restaurants.

In [ ]:
con.sql("""
    SELECT osm_id, tags['name'] AS name, tags['brand'] AS brand,
           tags['amenity'] AS amenity, tags['cuisine'] AS cuisine
    FROM raw.osm_food_places
    WHERE tags['brand:wikidata'] IS NULL
      AND (lower(tags['brand']) IN ('max', 'max burgers')
           OR lower(tags['name']) IN ('max', 'max burgers')
           OR lower(tags['name']) LIKE 'max %')
""")

**Finding:** only 3 of 7 are real, open Max restaurants.
- 2 × **Max Falafel** → a different restaurant.
- 2 × **"opens Dec 2026" / "opens late 2026"** → not open yet.

## 6. Places that are not open yet – any in the other chains?

In [ ]:
con.sql("""
    SELECT osm_id, tags['brand'] AS brand, tags['name'] AS name
    FROM raw.osm_food_places
    WHERE regexp_matches(lower(tags['name']), 'opens|öppnar|coming soon|kommer snart')
      AND regexp_matches(lower(coalesce(tags['brand'], tags['name'])),
                         'mcdonald|burger king|sibylla|max')
""")

**Finding:** fill in – how many, and which chains?

## 7. Preview of the final rule
This is the rule the dbt model `stg_osm__restaurants` will use:
1. match on `brand:wikidata`
2. no ID → match on brand/name, but not if it has a `shop` tag
3. drop places that are not open yet
4. drop places on a manual exclusion list (later a dbt seed)

In [ ]:
# manual exclusions: (osm_type, osm_id) – check the types in section 3 and 5
EXCLUDED = [
    ("node", 4326293367),  # Max Falafel
    ("node", 9128660434),  # Max Falafel
    ("node", 9061647857),  # Max clothing shop
]
excluded_sql = ", ".join(f"('{t}', {i})" for t, i in EXCLUDED)

con.sql(f"""
    WITH matched AS (
        SELECT
            osm_type, osm_id, tags,
            CASE tags['brand:wikidata']
                WHEN 'Q38076'   THEN 'mcdonalds'
                WHEN 'Q1912172' THEN 'max'
                WHEN 'Q177054'  THEN 'burger_king'
                WHEN 'Q488643'  THEN 'sibylla'
            END AS chain_by_id,
            CASE
                WHEN lower(coalesce(tags['brand'], tags['name'])) LIKE '%mcdonald%'
                  OR lower(coalesce(tags['brand'], tags['name'])) LIKE '%mc donald%' THEN 'mcdonalds'
                WHEN lower(coalesce(tags['brand'], tags['name'])) LIKE '%burger king%' THEN 'burger_king'
                WHEN lower(coalesce(tags['brand'], tags['name'])) LIKE '%sibylla%' THEN 'sibylla'
                WHEN lower(tags['brand']) IN ('max', 'max burgers')
                  OR lower(tags['name']) IN ('max', 'max burgers')
                  OR lower(tags['name']) LIKE 'max %' THEN 'max'
            END AS chain_by_name
        FROM raw.osm_food_places
    )
    SELECT
        coalesce(chain_by_id, chain_by_name) AS chain,
        count(*) AS restaurants,
        count(*) FILTER (WHERE chain_by_id IS NULL) AS via_fallback
    FROM matched
    WHERE (chain_by_id IS NOT NULL
           OR (tags['brand:wikidata'] IS NULL AND chain_by_name IS NOT NULL AND tags['shop'] IS NULL))
      AND NOT coalesce(regexp_matches(lower(tags['name']), 'opens|öppnar|coming soon|kommer snart'), false)
      AND (osm_type, osm_id) NOT IN (SELECT * FROM (VALUES {excluded_sql}))
    GROUP BY 1
    ORDER BY restaurants DESC
""")

**Finding:** fill in the final count per chain.

Expected roughly: McDonald's ~204, Max ~167, Sibylla ~121, Burger King ~115.

## Summary
- OSM has **15 483** food places in Sweden; extraction takes **28 s**.
- The four chains are matched on **`brand:wikidata`** (95 % of places), with a name fallback for the rest.
- Data-quality problems found: a clothing shop called Max, Max Falafel, and restaurants that are not open yet.
- They are handled by: no `shop` tag, an "opens" filter, and a manual exclusion list.

**Next:** compare the counts with each chain's own restaurant finder (OSM coverage check), then write the rule into `stg_osm__restaurants` in dbt.

In [ ]:
con.close()